## API to get data from NASA Website ##

In [1]:
# Using my own NASA MAP_KEY created on https://firms.modaps.eosdis.nasa.gov/api/map_key/
MAP_KEY = 'cce9957f998431b07b2b2501061fc6aa'


# now let's check how many transactions we have
import pandas as pd
import requests
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=cce9957f998431b07b2b2501061fc6aa'
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)



transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object

In [2]:
# let's create a simple function that tells us how many transactions we have used.
# We will use this in later examples

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

tcount = get_transaction_count()
print ('Our current transaction count is %i' % tcount)

Our current transaction count is 0


In [5]:
# let's query data_availability to find out what date range is available for various datasets
# we will explain these datasets a bit later

# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/cce9957f998431b07b2b2501061fc6aa/VIIRS_NOAA21_NRT'
df = pd.read_csv(da_url)
display(df)

,data_id,min_date,max_date
0,VIIRS_NOAA21_NRT,2024-01-17,2026-05-01


In [7]:
# in this example let's look at VIIRS NOAA-20, entire world and the most recent day
area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_NOAA21_NRT/world/1'
start_count = get_transaction_count()
df_area = pd.read_csv(area_url)
end_count = get_transaction_count()
print ('We used %i transactions.' % (end_count-start_count))

df_area

We used 36 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,53.39530,156.10124,355.66,0.61,0.71,2026-05-01,48,N21,VIIRS,l,2.0NRT,263.91,45.79,D
1,53.40085,156.09546,367.00,0.61,0.71,2026-05-01,48,N21,VIIRS,h,2.0NRT,293.53,99.15,D
2,53.40720,156.13222,355.66,0.60,0.71,2026-05-01,48,N21,VIIRS,l,2.0NRT,284.30,90.22,D
3,53.40976,156.11867,207.38,0.60,0.71,2026-05-01,48,N21,VIIRS,l,2.0NRT,279.40,388.00,D
4,53.41272,156.12639,367.00,0.60,0.71,2026-05-01,48,N21,VIIRS,h,2.0NRT,340.57,238.34,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11828,33.36269,-108.62203,295.76,0.60,0.50,2026-05-01,939,N21,VIIRS,n,2.0URT,267.39,1.29,N
11829,33.83680,-110.14281,297.72,0.40,0.50,2026-05-01,939,N21,VIIRS,n,2.0URT,278.06,4.02,N
11830,33.83780,-110.14745,335.67,0.40,0.50,2026-05-01,939,N21,VIIRS,n,2.0URT,279.77,6.08,N
11831,34.61965,-117.10004,303.53,0.40,0.40,2026-05-01,939,N21,VIIRS,n,2.0URT,286.70,1.10,N


## Chat GPT Code Visualization ##

In [2]:
pip install pandas folium


   -------------------------------- ------- 4/5 [folium]
   ---------------------------------------- 5/5 [folium]

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import folium

# Daten via API laden
MAP_KEY = "cce9957f998431b07b2b2501061fc6aa"

url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/VIIRS_NOAA21_NRT/world/1"

df = pd.read_csv(url)

print(df.head())


# Karte zentrieren (Mittelwert der Punkte)
m = folium.Map(
    location=[df.latitude.mean(), df.longitude.mean()],
    zoom_start=4
)

# Punkte hinzufügen
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color="red",
        fill=True,
        fill_opacity=0.7,
        popup=f"FRP: {row['frp']} MW\nConfidence: {row['confidence']}"
    ).add_to(m)

# speichern
m.save("fire_map.html")

   latitude  longitude  bright_ti4  scan  track    acq_date  acq_time  \
0  53.39530  156.10124      355.66  0.61   0.71  2026-05-01        48   
1  53.40085  156.09546      367.00  0.61   0.71  2026-05-01        48   
2  53.40720  156.13222      355.66  0.60   0.71  2026-05-01        48   
3  53.40976  156.11867      207.38  0.60   0.71  2026-05-01        48   
4  53.41272  156.12639      367.00  0.60   0.71  2026-05-01        48   

  satellite instrument confidence version  bright_ti5     frp daynight  
0       N21      VIIRS          l  2.0NRT      263.91   45.79        D  
1       N21      VIIRS          h  2.0NRT      293.53   99.15        D  
2       N21      VIIRS          l  2.0NRT      284.30   90.22        D  
3       N21      VIIRS          l  2.0NRT      279.40  388.00        D  
4       N21      VIIRS          h  2.0NRT      340.57  238.34        D  


In [6]:
def get_color(frp):
    if frp < 10:
        return "yellow"
    elif frp < 50:
        return "orange"
    else:
        return "red"

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=get_color(row["frp"]),
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

In [7]:
# nur verlässliche Feuer
df = df[df["confidence"] == "high"]

# oder nach Intensität filtern
df = df[df["frp"] > 10]

In [8]:
from folium.plugins import HeatMap

heat_data = df[["latitude", "longitude", "frp"]].values.tolist()

HeatMap(heat_data).add_to(m)